# BioRob Phase 1B — Final Organized Schema Guard / Pruner

**Purpose:** Scan every subject folder under:

`/home/tsultan1/BioRob/Human Subject Data/Sub-*/cleaned/`

and convert each Phase 1A cleaned CSV into the canonical BioRob schema used by the later synchronization/modeling pipeline.

This version is strict and paper-safe:
- Keeps `subject_id`, `task`, `trial`, and `Timestamp_seconds`.
- Does **not** silently remove those metadata columns.
- Checks all files for missing required columns.
- Checks subject/task/trial consistency.
- Makes optional backups before overwriting.
- Writes audit reports.

In [1]:
# ============================================================
# CELL 1 — Imports and final BioRob Phase 1B path settings
# ============================================================

from __future__ import annotations

from pathlib import Path
import json
import hashlib
import re
import shutil
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# FINAL BioRob path
ROOT_DIR = Path("/home/tsultan1/BioRob/Human Subject Data")

SUBJECT_GLOB = "Sub-*"
CLEANED_SUBDIR = "cleaned"

# Audit files are saved at the BioRob root level
AUDIT_DIR = ROOT_DIR / "_audit_phase1B_schema_prune"
REPORT_CSV = AUDIT_DIR / "phase1B_trainschema_prune_report.csv"
HEADER_REPORT_CSV = AUDIT_DIR / "phase1B_header_signature_report.csv"
SUBJECT_COUNT_CSV = AUDIT_DIR / "phase1B_subject_file_counts.csv"
SCHEMA_LOCK_JSON = AUDIT_DIR / "phase1B_train_schema.json"

# Safety settings
MAKE_BACKUP = True
BACKUP_SUBDIR_NAME = "_phase1B_backup_before_prune"

# Full run flag:
# False = only scan/check and show what would happen
# True  = actually overwrite cleaned CSV files with canonical Phase 1B columns
RUN_FULL_PHASE1B = True

print("ROOT_DIR:", ROOT_DIR)
print("ROOT exists:", ROOT_DIR.exists())
print("AUDIT_DIR:", AUDIT_DIR)
print("RUN_FULL_PHASE1B:", RUN_FULL_PHASE1B)
print("MAKE_BACKUP:", MAKE_BACKUP)

if not ROOT_DIR.exists():
    raise FileNotFoundError(f"ROOT_DIR not found: {ROOT_DIR}")

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
print("✅ PATH CHECK PASSED")


ROOT_DIR: /home/tsultan1/BioRob/Human Subject Data
ROOT exists: True
AUDIT_DIR: /home/tsultan1/BioRob/Human Subject Data/_audit_phase1B_schema_prune
RUN_FULL_PHASE1B: True
MAKE_BACKUP: True
✅ PATH CHECK PASSED


In [2]:
# ============================================================
# CELL 2 — Define expected Phase 1A columns and Phase 1B canonical output schema
# ============================================================

# These 55 columns are from your attached Phase 1A output file: 002_T516.csv
REFERENCE_PHASE1A_COLUMNS = [
    'subject_id', 'task', 'trial', 'Timestamp_seconds', 'Row', 'Timestamp', 'SampleNumber',
    'Ch1 EMG raw', 'Ch2 EMG raw', 'Ch3 EMG raw', 'Ch4 EMG raw',
    'ET_TimeSignal', 'ET_PupilLeft', 'ET_PupilRight', 'ET_DistanceLeft', 'ET_DistanceRight',
    'ET_GazeLeftx', 'ET_GazeLefty', 'ET_GazeRightx', 'ET_GazeRighty',
    'ET_ValidityLeftEye', 'ET_ValidityRightEye',
    'ET_GyroX', 'ET_GyroY', 'ET_GyroZ', 'ET_AccX', 'ET_AccY', 'ET_AccZ',
    'ET_HeadRotationPitch', 'ET_HeadRotationYaw', 'ET_HeadRotationRoll',
    'ET_Blink',
    'ET_Gaze3dEyeballXLeft', 'ET_Gaze3dEyeballYLeft', 'ET_Gaze3dEyeballZLeft',
    'ET_Gaze3dEyeballXRight', 'ET_Gaze3dEyeballYRight', 'ET_Gaze3dEyeballZRight',
    'ET_Gaze3dOpticalAxisXLeft', 'ET_Gaze3dOpticalAxisYLeft', 'ET_Gaze3dOpticalAxisZLeft',
    'ET_Gaze3dOpticalAxisXRight', 'ET_Gaze3dOpticalAxisYRight', 'ET_Gaze3dOpticalAxisZRight',
    'ET_Fixation', 'ET_Worn', 'LSL Timestamp',
    'Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch7', 'Ch8'
]

# Metadata columns: DO NOT REMOVE
META_COLS = ["subject_id", "task", "trial", "Timestamp_seconds"]

# EEG and EMG
EEG_COLS = [f"Ch{i}" for i in range(1, 9)]
EMG_COLS = [f"Ch{i} EMG raw" for i in range(1, 5)]

# Eye-tracking core and motion/head features used later
ET_CORE = [
    "ET_GazeLeftx", "ET_GazeLefty", "ET_GazeRightx", "ET_GazeRighty",
    "ET_PupilLeft", "ET_PupilRight",
    "ET_ValidityLeftEye", "ET_ValidityRightEye",
    "ET_Blink", "ET_Fixation", "ET_Worn",
]

ET_DIST = ["ET_DistanceLeft", "ET_DistanceRight"]

IMU_HEAD = [
    "ET_GyroX", "ET_GyroY", "ET_GyroZ",
    "ET_AccX", "ET_AccY", "ET_AccZ",
    "ET_HeadRotationPitch", "ET_HeadRotationYaw", "ET_HeadRotationRoll",
]

# Final Phase 1B output schema
CANONICAL_ORDER = META_COLS + EEG_COLS + EMG_COLS + ET_CORE + ET_DIST + IMU_HEAD

# These are required. If missing, the file is skipped, not silently fixed.
REQUIRED_COLS = set(CANONICAL_ORDER)

# These are intentionally removed from Phase 1B output because they are logging/time/geometry columns,
# not model/synchronization features.
DROP_COLS_EXPECTED = sorted(set(REFERENCE_PHASE1A_COLUMNS) - set(CANONICAL_ORDER))

print("Reference Phase 1A input column count:", len(REFERENCE_PHASE1A_COLUMNS))
print("Phase 1B canonical output column count:", len(CANONICAL_ORDER))
print("\nColumns that Phase 1B will KEEP:")
print(CANONICAL_ORDER)

print("\nColumns that Phase 1B will DROP if present:")
print(DROP_COLS_EXPECTED)

assert "subject_id" in CANONICAL_ORDER
assert "task" in CANONICAL_ORDER
assert "trial" in CANONICAL_ORDER
assert "Timestamp_seconds" in CANONICAL_ORDER

print("\n✅ SCHEMA DEFINED CORRECTLY — subject_id, task, trial, Timestamp_seconds are kept.")


Reference Phase 1A input column count: 55
Phase 1B canonical output column count: 38

Columns that Phase 1B will KEEP:
['subject_id', 'task', 'trial', 'Timestamp_seconds', 'Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch7', 'Ch8', 'Ch1 EMG raw', 'Ch2 EMG raw', 'Ch3 EMG raw', 'Ch4 EMG raw', 'ET_GazeLeftx', 'ET_GazeLefty', 'ET_GazeRightx', 'ET_GazeRighty', 'ET_PupilLeft', 'ET_PupilRight', 'ET_ValidityLeftEye', 'ET_ValidityRightEye', 'ET_Blink', 'ET_Fixation', 'ET_Worn', 'ET_DistanceLeft', 'ET_DistanceRight', 'ET_GyroX', 'ET_GyroY', 'ET_GyroZ', 'ET_AccX', 'ET_AccY', 'ET_AccZ', 'ET_HeadRotationPitch', 'ET_HeadRotationYaw', 'ET_HeadRotationRoll']

Columns that Phase 1B will DROP if present:
['ET_Gaze3dEyeballXLeft', 'ET_Gaze3dEyeballXRight', 'ET_Gaze3dEyeballYLeft', 'ET_Gaze3dEyeballYRight', 'ET_Gaze3dEyeballZLeft', 'ET_Gaze3dEyeballZRight', 'ET_Gaze3dOpticalAxisXLeft', 'ET_Gaze3dOpticalAxisXRight', 'ET_Gaze3dOpticalAxisYLeft', 'ET_Gaze3dOpticalAxisYRight', 'ET_Gaze3dOpticalAxisZLeft', 'ET_Gaz

In [ ]:
# ============================================================
# CELL 3 — Scan all Sub-* / cleaned folders
# ============================================================

def natural_sub_sort_key(path: Path):
    m = re.search(r"Sub-(\d+)$", path.name)
    return int(m.group(1)) if m else 10**9

def natural_file_sort_key(path: Path):
    # Sort by numeric prefix if present, then filename
    m = re.match(r"(\d+)_", path.name)
    return (int(m.group(1)) if m else 10**9, path.name)

subject_dirs = sorted(
    [p for p in ROOT_DIR.glob(SUBJECT_GLOB) if p.is_dir()],
    key=natural_sub_sort_key
)

print("Subjects found:", len(subject_dirs))
print([p.name for p in subject_dirs])

all_csvs = []
subject_rows = []

for subj_dir in subject_dirs:
    cleaned_dir = subj_dir / CLEANED_SUBDIR
    csvs = sorted(cleaned_dir.glob("*.csv"), key=natural_file_sort_key) if cleaned_dir.exists() else []
    all_csvs.extend(csvs)
    subject_rows.append({
        "subject": subj_dir.name,
        "cleaned_dir": str(cleaned_dir),
        "cleaned_dir_exists": cleaned_dir.exists(),
        "csv_count": len(csvs),
    })

subject_count_df = pd.DataFrame(subject_rows)
display(subject_count_df)

subject_count_df.to_csv(SUBJECT_COUNT_CSV, index=False)
print("\nSaved subject count report:", SUBJECT_COUNT_CSV)
print("Total CSV files found:", len(all_csvs))

if len(all_csvs) == 0:
    raise FileNotFoundError("No CSV files found under Sub-*/cleaned/. Check ROOT_DIR and CLEANED_SUBDIR.")

print("\n✅ SUBJECT SCAN COMPLETED")


In [ ]:
# ============================================================
# CELL 4 — Header signature check before modifying files
# This confirms whether all Phase 1A outputs have the same columns.
# ============================================================

def read_header_only(csv_path: Path) -> list[str]:
    return list(pd.read_csv(csv_path, nrows=0, encoding="utf-8-sig").columns)

def header_sig(cols: list[str]) -> str:
    return hashlib.md5("|".join(cols).encode("utf-8")).hexdigest()[:10]

header_rows = []
signature_to_cols = {}

for csv_path in all_csvs:
    try:
        cols = read_header_only(csv_path)
        sig = header_sig(cols)
        signature_to_cols.setdefault(sig, cols)
        header_rows.append({
            "subject": csv_path.parents[1].name,
            "file": csv_path.name,
            "n_cols": len(cols),
            "header_sig": sig,
            "matches_attached_phase1A_55cols": cols == REFERENCE_PHASE1A_COLUMNS,
            "already_phase1B_canonical_38cols": cols == CANONICAL_ORDER,
            "missing_required": ";".join([c for c in CANONICAL_ORDER if c not in cols]),
            "extra_vs_canonical": ";".join([c for c in cols if c not in CANONICAL_ORDER]),
        })
    except Exception as e:
        header_rows.append({
            "subject": csv_path.parents[1].name,
            "file": csv_path.name,
            "n_cols": None,
            "header_sig": "READ_FAIL",
            "matches_attached_phase1A_55cols": False,
            "already_phase1B_canonical_38cols": False,
            "missing_required": "READ_FAIL",
            "extra_vs_canonical": "",
            "error": repr(e),
        })

header_df = pd.DataFrame(header_rows)
header_df.to_csv(HEADER_REPORT_CSV, index=False)

print("Unique header signatures:", header_df["header_sig"].nunique())
print("Header report saved:", HEADER_REPORT_CSV)

summary = (
    header_df
    .groupby(["header_sig", "n_cols", "matches_attached_phase1A_55cols", "already_phase1B_canonical_38cols"], dropna=False)
    .size()
    .reset_index(name="file_count")
    .sort_values("file_count", ascending=False)
)
display(summary)

bad_missing = header_df[
    (header_df["missing_required"].fillna("") != "") &
    (header_df["missing_required"].fillna("") != "READ_FAIL")
]

if len(bad_missing) > 0:
    print("\n❌ Some files are missing required Phase 1B columns. Showing first 20:")
    display(bad_missing.head(20))
    print("Do NOT continue until you inspect the header report.")
else:
    print("\n✅ HEADER CHECK PASSED — all files have the required columns.")


In [5]:
# ============================================================
# CELL 5 — Filename and metadata consistency helpers
# Checks subject_id, task, and trial without removing them.
# ============================================================

def parse_subject_from_folder(csv_path: Path):
    # Expected: .../Sub-1/cleaned/file.csv
    for part in csv_path.parts:
        m = re.fullmatch(r"Sub-(\d+)", part)
        if m:
            return int(m.group(1))
    return None

def parse_task_trial_from_filename(csv_path: Path):
    # Examples:
    # 002_T516.csv -> task=5, trial=16
    # 067_T116.csv -> task=1, trial=16
    # First digit after T/M is task; remaining digits are trial.
    stem = csv_path.stem
    m = re.search(r"[_-]([TM])(\d+)$", stem, flags=re.IGNORECASE)
    if not m:
        return None, None, None
    mode = m.group(1).upper()
    digits = m.group(2)
    if len(digits) < 2:
        return mode, None, None
    task = int(digits[0])
    trial = int(digits[1:])
    return mode, task, trial

def safe_unique_nonnull(series: pd.Series, max_show=10):
    vals = pd.Series(series).dropna().unique()
    if len(vals) > max_show:
        return list(vals[:max_show]) + ["..."]
    return list(vals)

# Demo on first few files
for p in all_csvs[:5]:
    folder_sub = parse_subject_from_folder(p)
    mode, task_from_name, trial_from_name = parse_task_trial_from_filename(p)
    print(p)
    print("  folder subject:", folder_sub, "| filename mode/task/trial:", mode, task_from_name, trial_from_name)

print("\n✅ PARSER READY")


/home/tsultan1/BioRob/Human Subject Data/Sub-1/cleaned/001_T03.csv
  folder subject: 1 | filename mode/task/trial: T 0 3
/home/tsultan1/BioRob/Human Subject Data/Sub-1/cleaned/002_T516.csv
  folder subject: 1 | filename mode/task/trial: T 5 16
/home/tsultan1/BioRob/Human Subject Data/Sub-1/cleaned/003_T515.csv
  folder subject: 1 | filename mode/task/trial: T 5 15
/home/tsultan1/BioRob/Human Subject Data/Sub-1/cleaned/004_T514.csv
  folder subject: 1 | filename mode/task/trial: T 5 14
/home/tsultan1/BioRob/Human Subject Data/Sub-1/cleaned/005_T513.csv
  folder subject: 1 | filename mode/task/trial: T 5 13

✅ PARSER READY


In [6]:
# ============================================================
# CELL 6 — Core Phase 1B prune/validate functions
# ============================================================

NUMERIC_COLS = set(CANONICAL_ORDER)

def read_csv_safely(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="utf-8-sig", on_bad_lines="skip")
    except Exception:
        return pd.read_csv(path, encoding="utf-8-sig", engine="python", on_bad_lines="skip")

def coerce_phase1B_types(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Coerce all canonical columns to numeric.
    # This keeps subject_id/task/trial columns but makes sure they are numeric.
    for c in CANONICAL_ORDER:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    # Convert metadata columns to nullable integer where possible.
    for c in ["subject_id", "task", "trial"]:
        if out[c].notna().all():
            out[c] = out[c].astype("int64")

    return out

def check_metadata_consistency(df: pd.DataFrame, csv_path: Path) -> dict:
    folder_subject = parse_subject_from_folder(csv_path)
    mode, file_task, file_trial = parse_task_trial_from_filename(csv_path)

    info = {
        "folder_subject": folder_subject,
        "filename_mode": mode,
        "filename_task": file_task,
        "filename_trial": file_trial,
        "subject_unique": safe_unique_nonnull(df["subject_id"]) if "subject_id" in df.columns else [],
        "task_unique": safe_unique_nonnull(df["task"]) if "task" in df.columns else [],
        "trial_unique": safe_unique_nonnull(df["trial"]) if "trial" in df.columns else [],
        "metadata_warning": "",
    }

    warnings = []

    if "subject_id" in df.columns and folder_subject is not None:
        subj_vals = pd.to_numeric(df["subject_id"], errors="coerce").dropna().unique()
        if len(subj_vals) != 1 or int(subj_vals[0]) != folder_subject:
            warnings.append(f"subject_id values {list(subj_vals[:10])} do not match folder Sub-{folder_subject}")

    if "task" in df.columns and file_task is not None:
        task_vals = pd.to_numeric(df["task"], errors="coerce").dropna().unique()
        if len(task_vals) != 1 or int(task_vals[0]) != file_task:
            warnings.append(f"task values {list(task_vals[:10])} do not match filename task {file_task}")

    if "trial" in df.columns and file_trial is not None:
        trial_vals = pd.to_numeric(df["trial"], errors="coerce").dropna().unique()
        if len(trial_vals) != 1 or int(trial_vals[0]) != file_trial:
            warnings.append(f"trial values {list(trial_vals[:10])} do not match filename trial {file_trial}")

    info["metadata_warning"] = " | ".join(warnings)
    return info

def make_backup_once(csv_path: Path) -> str:
    backup_dir = csv_path.parent / BACKUP_SUBDIR_NAME
    backup_dir.mkdir(parents=True, exist_ok=True)
    backup_path = backup_dir / csv_path.name

    if backup_path.exists():
        return f"backup_exists:{backup_path}"

    shutil.copy2(csv_path, backup_path)
    return f"backup_created:{backup_path}"

def prune_one_file(csv_path: Path, dry_run: bool = True) -> dict:
    result = {
        "subject": csv_path.parents[1].name,
        "file": csv_path.name,
        "path": str(csv_path),
        "status": "",
        "rows_in": None,
        "cols_in": None,
        "rows_out": None,
        "cols_out": None,
        "missing_required": "",
        "dropped_cols": "",
        "kept_cols": "",
        "metadata_warning": "",
        "backup_status": "",
        "schema_sig_out": "",
        "message": "",
    }

    try:
        df = read_csv_safely(csv_path)
        result["rows_in"] = len(df)
        result["cols_in"] = df.shape[1]

        missing_required = [c for c in CANONICAL_ORDER if c not in df.columns]
        result["missing_required"] = ";".join(missing_required)

        if missing_required:
            result["status"] = "SKIPPED_MISSING_REQUIRED"
            result["message"] = "File was not overwritten because required columns are missing."
            return result

        meta_info = check_metadata_consistency(df, csv_path)
        result["metadata_warning"] = meta_info["metadata_warning"]

        # Keep canonical columns only, in canonical order.
        pruned = df.loc[:, CANONICAL_ORDER].copy()
        pruned = coerce_phase1B_types(pruned)

        dropped_cols = [c for c in df.columns if c not in CANONICAL_ORDER]
        result["dropped_cols"] = ";".join(dropped_cols)
        result["kept_cols"] = ";".join(CANONICAL_ORDER)
        result["rows_out"] = len(pruned)
        result["cols_out"] = pruned.shape[1]
        result["schema_sig_out"] = hashlib.md5("|".join(pruned.columns).encode("utf-8")).hexdigest()[:10]

        # Critical safety check: metadata columns remain in output.
        for required_meta in ["subject_id", "task", "trial", "Timestamp_seconds"]:
            if required_meta not in pruned.columns:
                raise RuntimeError(f"Critical metadata column removed: {required_meta}")

        # Do not overwrite if there is a subject/task/trial mismatch.
        if result["metadata_warning"]:
            result["status"] = "SKIPPED_METADATA_MISMATCH"
            result["message"] = "File was not overwritten because metadata does not match folder/filename."
            return result

        if dry_run:
            result["status"] = "DRY_RUN_OK"
            result["message"] = "Dry run only; file not modified."
            return result

        if MAKE_BACKUP:
            result["backup_status"] = make_backup_once(csv_path)

        pruned.to_csv(csv_path, index=False, encoding="utf-8-sig")
        result["status"] = "SAVED_OK"
        result["message"] = "✅ CLEANING APPLIED CORRECTLY — Phase 1B schema saved."
        return result

    except Exception as e:
        result["status"] = "ERROR"
        result["message"] = repr(e)
        return result

print("✅ CORE FUNCTIONS LOADED")


✅ CORE FUNCTIONS LOADED


In [7]:
# ============================================================
# CELL 7 — Single-file test on the first CSV before batch run
# This prints exactly what Phase 1B will keep/drop.
# ============================================================

test_csv = all_csvs[0]
print("Testing first CSV:")
print(test_csv)

single_result = prune_one_file(test_csv, dry_run=True)
for k, v in single_result.items():
    if k in ["kept_cols", "dropped_cols"]:
        print(f"{k}:")
        print(v)
    else:
        print(f"{k}: {v}")

if single_result["status"] == "DRY_RUN_OK":
    print("\n✅ SINGLE-FILE PHASE 1B TEST PASSED — subject_id, task, trial are preserved.")
else:
    print("\n❌ SINGLE-FILE TEST DID NOT PASS. Check message above before full run.")


Testing first CSV:
/home/tsultan1/BioRob/Human Subject Data/Sub-1/cleaned/001_T03.csv
subject: Sub-1
file: 001_T03.csv
path: /home/tsultan1/BioRob/Human Subject Data/Sub-1/cleaned/001_T03.csv
status: DRY_RUN_OK
rows_in: 16787
cols_in: 55
rows_out: 16787
cols_out: 38
missing_required: 
dropped_cols:
Row;Timestamp;SampleNumber;ET_TimeSignal;ET_Gaze3dEyeballXLeft;ET_Gaze3dEyeballYLeft;ET_Gaze3dEyeballZLeft;ET_Gaze3dEyeballXRight;ET_Gaze3dEyeballYRight;ET_Gaze3dEyeballZRight;ET_Gaze3dOpticalAxisXLeft;ET_Gaze3dOpticalAxisYLeft;ET_Gaze3dOpticalAxisZLeft;ET_Gaze3dOpticalAxisXRight;ET_Gaze3dOpticalAxisYRight;ET_Gaze3dOpticalAxisZRight;LSL Timestamp
kept_cols:
subject_id;task;trial;Timestamp_seconds;Ch1;Ch2;Ch3;Ch4;Ch5;Ch6;Ch7;Ch8;Ch1 EMG raw;Ch2 EMG raw;Ch3 EMG raw;Ch4 EMG raw;ET_GazeLeftx;ET_GazeLefty;ET_GazeRightx;ET_GazeRighty;ET_PupilLeft;ET_PupilRight;ET_ValidityLeftEye;ET_ValidityRightEye;ET_Blink;ET_Fixation;ET_Worn;ET_DistanceLeft;ET_DistanceRight;ET_GyroX;ET_GyroY;ET_GyroZ;ET_AccX;ET_

In [ ]:
# ============================================================
# CELL 8 — Run Phase 1B on all Sub-* / cleaned CSV files
# ============================================================

dry_run = not RUN_FULL_PHASE1B

print("RUN_FULL_PHASE1B:", RUN_FULL_PHASE1B)
print("dry_run:", dry_run)
print("Total files to process:", len(all_csvs))

results = []

for i, csv_path in enumerate(all_csvs, start=1):
    res = prune_one_file(csv_path, dry_run=dry_run)
    results.append(res)

    # Print every file because this is final verification.
    print(
        f"[{i:04d}/{len(all_csvs):04d}] {res['subject']}/{res['file']} | "
        f"{res['status']} | rows {res['rows_in']} -> {res['rows_out']} | "
        f"cols {res['cols_in']} -> {res['cols_out']}"
    )

    if res["metadata_warning"]:
        print("   ⚠️", res["metadata_warning"])
    if res["message"] and res["status"] not in ["SAVED_OK", "DRY_RUN_OK"]:
        print("   message:", res["message"])

report_df = pd.DataFrame(results)
report_df.to_csv(REPORT_CSV, index=False)

print("\nSaved Phase 1B report:", REPORT_CSV)
print("\nStatus counts:")
display(report_df["status"].value_counts(dropna=False).to_frame("count"))

bad = report_df[~report_df["status"].isin(["SAVED_OK", "DRY_RUN_OK"])]
if len(bad) > 0:
    print("\n❌ Some files were not processed correctly. Showing first 30:")
    display(bad.head(30))
else:
    print("\n✅ PHASE 1B COMPLETED SUCCESSFULLY FOR ALL FILES")


In [ ]:
# ============================================================
# CELL 9 — Post-run verification: re-read saved files and confirm final schema
# ============================================================

verify_rows = []

for csv_path in all_csvs:
    try:
        cols = read_header_only(csv_path)
        verify_rows.append({
            "subject": csv_path.parents[1].name,
            "file": csv_path.name,
            "n_cols_after": len(cols),
            "schema_exact": cols == CANONICAL_ORDER,
            "missing_after": ";".join([c for c in CANONICAL_ORDER if c not in cols]),
            "extra_after": ";".join([c for c in cols if c not in CANONICAL_ORDER]),
            "has_subject_id": "subject_id" in cols,
            "has_task": "task" in cols,
            "has_trial": "trial" in cols,
            "has_Timestamp_seconds": "Timestamp_seconds" in cols,
        })
    except Exception as e:
        verify_rows.append({
            "subject": csv_path.parents[1].name,
            "file": csv_path.name,
            "n_cols_after": None,
            "schema_exact": False,
            "missing_after": "READ_FAIL",
            "extra_after": "",
            "has_subject_id": False,
            "has_task": False,
            "has_trial": False,
            "has_Timestamp_seconds": False,
            "error": repr(e),
        })

verify_df = pd.DataFrame(verify_rows)
verify_csv = AUDIT_DIR / "phase1B_postrun_schema_verification.csv"
verify_df.to_csv(verify_csv, index=False)

print("Post-run verification saved:", verify_csv)
print("\nSchema exact counts:")
display(verify_df["schema_exact"].value_counts(dropna=False).to_frame("count"))

meta_ok = (
    verify_df["has_subject_id"].all()
    and verify_df["has_task"].all()
    and verify_df["has_trial"].all()
    and verify_df["has_Timestamp_seconds"].all()
)

if verify_df["schema_exact"].all() and meta_ok:
    print("\n✅ CLEANING APPLIED CORRECTLY")
    print("✅ All saved CSVs have the exact Phase 1B canonical schema.")
    print("✅ subject_id, task, trial, and Timestamp_seconds were NOT removed.")
else:
    print("\n❌ CLEANING NOT FULLY APPLIED")
    print("Some files do not match the final schema. Showing first 30 problematic files:")
    display(verify_df[~verify_df["schema_exact"]].head(30))


In [10]:
# ============================================================
# CELL 10 — Inspect one saved CSV after Phase 1B
# ============================================================

inspect_csv = all_csvs[0]
saved_df = pd.read_csv(inspect_csv, nrows=10)

print("Inspecting saved file:")
print(inspect_csv)
print("Shape preview:", saved_df.shape)
print("\nColumns:")
print(list(saved_df.columns))

print("\nFirst 10 rows:")
display(saved_df)

needed = ["subject_id", "task", "trial", "Timestamp_seconds"]
if all(c in saved_df.columns for c in needed):
    print("\n✅ Metadata columns are present:", needed)
else:
    print("\n❌ Metadata columns missing:", [c for c in needed if c not in saved_df.columns])


Inspecting saved file:
/home/tsultan1/BioRob/Human Subject Data/Sub-1/cleaned/001_T03.csv
Shape preview: (10, 38)

Columns:
['subject_id', 'task', 'trial', 'Timestamp_seconds', 'Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch7', 'Ch8', 'Ch1 EMG raw', 'Ch2 EMG raw', 'Ch3 EMG raw', 'Ch4 EMG raw', 'ET_GazeLeftx', 'ET_GazeLefty', 'ET_GazeRightx', 'ET_GazeRighty', 'ET_PupilLeft', 'ET_PupilRight', 'ET_ValidityLeftEye', 'ET_ValidityRightEye', 'ET_Blink', 'ET_Fixation', 'ET_Worn', 'ET_DistanceLeft', 'ET_DistanceRight', 'ET_GyroX', 'ET_GyroY', 'ET_GyroZ', 'ET_AccX', 'ET_AccY', 'ET_AccZ', 'ET_HeadRotationPitch', 'ET_HeadRotationYaw', 'ET_HeadRotationRoll']

First 10 rows:


,subject_id,task,trial,Timestamp_seconds,Ch1,Ch2,Ch3,Ch4,Ch5,Ch6,Ch7,Ch8,Ch1 EMG raw,Ch2 EMG raw,Ch3 EMG raw,Ch4 EMG raw,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_PupilLeft,ET_PupilRight,ET_ValidityLeftEye,ET_ValidityRightEye,ET_Blink,ET_Fixation,ET_Worn,ET_DistanceLeft,ET_DistanceRight,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll
0,1,0,3,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72.0,-20.0,64.0,-152.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0,3,0.002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-26.0,-78.0,719.0,-1232.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,0,3,0.004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,-20.0,358.0,108.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,0,3,0.006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.0,-104.0,-229.0,-626.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,0,3,0.008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-39.0,-46.0,-600.0,-392.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1,0,3,0.010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-160.0,253.0,196.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,0,3,0.012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-105.0,-90.0,272.0,208.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1,0,3,0.014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5.0,-28.0,92.0,704.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1,0,3,0.016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37.0,44.0,104.0,-162.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1,0,3,0.018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,2.0,-140.0,-3481.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



✅ Metadata columns are present: ['subject_id', 'task', 'trial', 'Timestamp_seconds']


In [11]:
# ============================================================
# CELL 11 — Write schema lock JSON for later phases
# ============================================================

schema_lock = {
    "phase": "Phase 1B",
    "root_dir": str(ROOT_DIR),
    "cleaned_subdir": CLEANED_SUBDIR,
    "meta_cols_kept": META_COLS,
    "eeg_cols": EEG_COLS,
    "emg_cols": EMG_COLS,
    "et_core": ET_CORE,
    "et_dist": ET_DIST,
    "imu_head": IMU_HEAD,
    "canonical_order": CANONICAL_ORDER,
    "n_canonical_cols": len(CANONICAL_ORDER),
    "reference_phase1A_columns_from_uploaded_002_T516": REFERENCE_PHASE1A_COLUMNS,
    "n_reference_phase1A_cols": len(REFERENCE_PHASE1A_COLUMNS),
    "dropped_from_phase1A_if_present": DROP_COLS_EXPECTED,
    "notes": [
        "subject_id, task, trial, and Timestamp_seconds are kept.",
        "Files missing required canonical columns are skipped, not silently fixed.",
        "Phase 1B output is intentionally reduced from the Phase 1A cleaned 55-column schema to the 38-column canonical schema.",
    ],
}

with open(SCHEMA_LOCK_JSON, "w", encoding="utf-8") as f:
    json.dump(schema_lock, f, indent=2)

print("Schema lock saved:", SCHEMA_LOCK_JSON)
print("\n✅ PHASE 1B NOTEBOOK FINISHED")


Schema lock saved: /home/tsultan1/BioRob/Human Subject Data/_audit_phase1B_schema_prune/phase1B_train_schema.json

✅ PHASE 1B NOTEBOOK FINISHED


## Expected final result

After successful Phase 1B:

- Each `Sub-*/cleaned/*.csv` should have **38 columns**.
- These metadata columns must still exist:
  - `subject_id`
  - `task`
  - `trial`
  - `Timestamp_seconds`
- Audit outputs are saved in:

`/home/tsultan1/BioRob/Human Subject Data/_audit_phase1B_schema_prune/`

Then continue to **Phase 1C synchronization**.